In [ ]:
import torch
import torch.nn as nn

import rosbag2_py
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message
import pandas as pd

import numpy as np
from scipy.spatial.transform import Rotation as R

bag_path = ""

In [ ]:
def get_data(bag_path):
    reader = rosbag2_py.SequentialReader()
    storage_options = rosbag2_py.StorageOptions(uri=bag_path, storage_id='sqlite3')
    converter_options = rosbag2_py.ConverterOptions(input_serialization_format='cdr', output_serialization_format='cdr')
    reader.open(storage_options, converter_options)

    topic_types = reader.get_all_topics_and_types()
    type_map = {topic.name: topic.type for topic in topic_types}

    rows = []
    while reader.has_next():
        (topic, data, t) = reader.read_next()
        msg_type = get_message(type_map[topic])
        msg = deserialize_message(data, msg_type)

        if topic == '/imu':
            rows.append({'time': t, 'type': 'imu', 'val': [msg.linear_acceleration.x, msg.linear_acceleration.y, msg.linear_acceleration.z, msg.angular_velocity.x, msg.angular_velocity.y, msg.angular_velocity.z]})
        elif topic == '/scan':
            rows.append({'time': t, 'type': 'scan', 'val': list(msg.ranges)})
        elif topic == '/odom':
            rows.append({'time': t, 'type': 'odom', 'val': [msg.pose.pose.position.x, msg.pose.pose.position.y]})
        elif topic == "/pose":
            ori = msg.pose.orientation
            pos = msg.pose.position
            rows.append({'time': msg.header.stamp.sec + (msg.header.stamp.nanosec * 1e-9), 'type': 'pose', 'val': [pos.x, pos.y, pos.z, ori.x, ori.y, ori.z, ori.w]})

    return pd.DataFrame(rows)

In [ ]:
df = get_data(bag_path)

df_scan = df[df['type'] == 'scan'].copy()
df_imu = df[df['type'] == 'imu'].copy()

df_odom = df[df['type'] == 'odom'].copy()
df_odom[['x', 'y']] = pd.DataFrame(df_odom['val'].tolist(), index=df_odom.index)
df_odom[['x', 'y']] = df_odom[['x', 'y']] - df_odom[['x', 'y']].iloc[0]

df_gt = df[df['type'] == 'pose'].copy()

In [ ]:
df_scan = df_scan.sort_values("time")
df_imu = df_imu.sort_values("time")
df_odom = df_odom.sort_values("time")
df_gt = df_gt.sort_values("time")

synced_df = pd.merge_asof(df_scan, df_imu, on = 'time', direction = "nearest", suffixes= ("_scan", "_imu") )
synced_df = pd.merge_asof(synced_df, df_odom, on = 'time', direction = "nearest", suffixes= ("", "_odom") )
synced_df = pd.merge_asof(synced_df, df_gt, on = 'time', direction = "nearest", suffixes= ("", "_gt") )

synced_df.dropna()

The data will be offset depending on where the sensors were placed on the robot. This data is stored in the /tf_static topic. We will get the static_tf data and later use it to center our sensor data.

In [ ]:
from tf2_msgs.msg import TFMessage

def get_static_tf(bag_path):
    reader = rosbag2_py.SequentialReader()
    storage_options = rosbag2_py.StorageOptions(uri=bag_path, storage_id='sqlite3')
    converter_options = rosbag2_py.ConverterOptions(input_serialization_format='cdr', output_serialization_format='cdr')
    reader.open(storage_options, converter_options)

    while reader.has_next():
        (topic, data, t) = reader.read_next()
        if topic == '/tf_static':
            msg = deserialize_message(data, TFMessage)
            for transform in msg.transforms:
                if transform.child_frame_id == 'rplidar_link':
                    t = transform.transform.translation
                    r = transform.transform.rotation
                    print(f"Laser Offset: x={t.x}, y={t.y}, z={t.z}")
                    print(f"Laser Quat: x={r.x}, y={r.y}, z={r.z}, w={r.w}")
                if transform.child_frame_id == 'imu_link':
                    t = transform.transform.translation
                    r = transform.transform.rotation
                    print(f"IMU Offset: x={t.x}, y={t.y}, z={t.z}")
                    print(f"IMU Quat: x={r.x}, y={r.y}, z={r.z}, w={r.w}")
            break 

In [ ]:
get_static_tf(bag_path)

In [ ]:

LASER_OFFSET_X,LASER_OFFSET_Y,LASER_OFFSET_Z  = 0.1,0.0,0.0 # replace values with the printed values of "Laser Offset: ..."
LASER_ROT = [0, 0, 0, 1] # replace values with the printed values of "Laser Quat: ..."

IMU_OFFSET_X,IMU_OFFSET_Y,IMU_OFFSET_Z  = 0.1,0.0,0.0 # replace values with the printed values of "IMU Offset: ..."
IMU_ROT = [0, 0, 0, 1] # replace values with the printed values of "IMU Quat: ..."

def clean_sensor_data(raw_imu, raw_scan):
    cleaned_imu = []
    cleaned_scan = []

    imu_rot = R.from_quat(IMU_ROT)
    imu_os = np.array([IMU_OFFSET_X,IMU_OFFSET_Y,IMU_OFFSET_Z])
    
    for val in raw_imu:
        accel = np.array(val[:3])
        gyro = np.array(val[3:])  
        
        cr_accel = imu_rot.apply(accel)
        cr_gyro = imu_rot.apply(gyro)

        lever_arm_accel = np.cross(cr_gyro, np.cross(cr_gyro, imu_os))

        cleaned_imu.append(np.concatenate([cr_accel - lever_arm_accel, cr_gyro]))

    num_scan = len(raw_scan[0])
    angles = np.linspace(0, 2 * np.pi, num_scan)
    laser_rot = R.from_quat(LASER_ROT)

    for val in raw_scan:
        ranges = np.array(val)
        ranges = np.where(np.isinf(ranges), 12.0, ranges) 

        sensor_points = np.zeros((num_scan,3))
        sensor_points[:,0] = ranges * np.cos(angles)
        sensor_points[:,1] = ranges * np.sin(angles)
        sensor_points[:,2] = 0

        cr_points = laser_rot.apply(sensor_points)
        fixed_ranges = np.sqrt((cr_points[:,0] + LASER_OFFSET_X)**2 + (cr_points[:,1] + LASER_OFFSET_Y)**2 + (cr_points[:,2] + LASER_OFFSET_Z)**2)
        
        cleaned_scan.append(fixed_ranges)

    return np.array(cleaned_imu), np.array(cleaned_scan)

In [ ]:
#?????????????

In [ ]:



imu_list = [ row.val for row in df_imu.itertuples()]
scan_list = [ row.val for row in df_scan.itertuples()]

c_imu, c_scan = clean_sensor_data(imu_list,scan_list)



